# L09 · Viewing an LLM as a Policy

## Goal

- interpret tokens as actions
- distinguish scalar and verifiable rewards
- explain the role of reference KL

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L09:toy:42").hexdigest()
print(f"lesson=L09 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L09 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:07619f45c3d0f957828627af750a847b0eb5fcb65e4586c2552b7c6beec9efae data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: PPO + causal LM → **LLM policy, reward, and reference KL** → RLHF/DPO/GRPO

$$r_t^{total}=r_t^{task}-\beta\left(\log\pi_\theta(a_t)-\log\pi_{ref}(a_t)\right)$$

The prompt is the initial state, generated tokens are actions, and each prefix is the next state. Scalar reward often lands at response end, while KL shaping can be computed per action token. A reference model measures rapid drift from the SFT policy.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** What should KL and token reward be at the third position where the action mask is false? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>Both must be zero. Prompt, padding, and tool-output tokens are not actions selected by the policy.</details>

In [2]:
from rl_study.algorithms.rlhf_ppo import compose_rlhf_rewards
policy_logp = torch.tensor([[-0.2, -0.3, 0.0]])
reference_logp = torch.tensor([[-0.3, -0.25, 0.0]])
token_action_mask = torch.tensor([[True, True, False]])
reward_parts = compose_rlhf_rewards(
    torch.tensor([1.0]), policy_logp, reference_logp,
    token_action_mask, kl_coefficient=0.1
)
print({"sampled_kl": reward_parts.sampled_kl.tolist(),
       "token_rewards": reward_parts.token_rewards.tolist(),
       "total_reward": reward_parts.total_rewards.tolist()})

{'sampled_kl': [[0.10000000894069672, -0.050000011920928955, 0.0]], 'token_rewards': [[-0.010000000707805157, 1.0049999952316284, -0.0]], 'total_reward': [0.9950000047683716]}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Sampled KL is a fast selected-token log-ratio, but it has variance and can be negative. Full-distribution KL is costlier and answers a different diagnostic question.

**Common trap:** Repeating scalar reward on every token duplicates it by response length. Attach it once at the terminal action and audit it separately from KL shaping. Regression tests: `test_rlhf_reward_plus_token_kl_decomposition`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert reward_parts.token_rewards[0, 2].item() == 0.0
assert torch.allclose(reward_parts.total_rewards, reward_parts.token_rewards.sum(-1))
print("checks=passed")

checks=passed


**Recall:** Why does one negative sampled-KL term not prove the regularizer is wrong? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** The third masked position is zero, and the two action-token rewards sum to 0.995. Task reward and KL cost remain separately inspectable.
- Executable checks: `test_rlhf_reward_plus_token_kl_decomposition`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L10 connects SFT, reward source, rollout, and PPO update into one lifecycle.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

[Implementation note](../../docs/algorithms/llm-foundations.md) · [Course map](../../docs/course-map.en.md)

## Sources

- `learning-to-summarize-2020` — `docs/sources.yml`
- `instructgpt-2022` — `docs/sources.yml`